# Harmonia — population-of-models: physiological variability (HYPOTHESIS-TIER)

Where [`01_flip_frequency`](01_flip_frequency.ipynb) propagates **input** (IC50)
variability into a risk distribution, this notebook propagates **physiological**
(between-heart) variability. Instead of one ventricular myocyte, `assess_population`
builds a *population* of virtual myocytes by sampling the kernel conductances, then
runs the drug across all of them — yielding a spread of risk classifications and a
**susceptible fraction**.

It covers the three population layers Harmonia ships:

1. the v0.1 **variability cloud** (`illustrative_v0`),
2. the v0.3 **disease/genetic backgrounds** (the LQTS channelopathies — a per-current
   *mean* shift that spends repolarization reserve), and
3. the v0.5 **experimentally-calibrated** population (Britton et al. 2013 — admit only
   drug-free-plausible myocytes).

> **NON-NEGOTIABLE (spec.md §3, §10).** The conductance variability and any disease
> mean-shift are *illustrative*, **not** calibrated to patient data, and the
> calibration ranges are *kernel-plausibility* bounds (not a fit to human
> electrophysiology). Every population assessment is therefore capped at **Tier D**
> and stamped **NOT FOR PREDICTION**. This is a hypothesis-generating methodology
> view, never a per-patient or per-genotype safety claim.

Like the other notebooks, it is **deterministic (seeded)** and carries inline
assertions, so a clean run under `nbmake` is a test, not just a demo.

In [ ]:
import harmonia
from harmonia.populations import assess_population, calibrate_population

ds = harmonia.load()
print(f"harmonia {harmonia.__version__}  —  populations:",
      ", ".join(p.id.split('.', 1)[1] for p in ds.populations))

## 1. The variability cloud — one drug, a spread of hearts

The healthy population samples each conductance lognormally (per-channel CVs from the
`illustrative_v0` record). A drug that the single-cell estimate "misses" can still have
a substantial susceptible subpopulation — exactly the sensitivity gain population
approaches are known for. **sotalol** (weak-hERG at 4× EFTPC) is the worked example.

In [ ]:
pop = assess_population(ds, "sotalol", population="illustrative_v0", n_models=60, seed=0)
print(pop.summary())

# Guardrail: a population assessment is ALWAYS Tier D and never a prediction.
assert pop.tier == "D"
assert "NOT FOR PREDICTION" in pop.summary()
# A spread, not a point: some myocytes cross into high risk, others do not.
assert 0.0 < pop.susceptible_fraction < 1.0

## 2. Disease backgrounds — repolarization reserve, spent

A congenital long-QT channelopathy is a systematic *mean* shift of one current
([Moss & Kass 2005](https://doi.org/10.1172/JCI25537)): `lqt1` (IKs↓), `lqt2` (IKr↓),
`lqt3` (late-INa↑). The healthy ventricle repolarizes through redundant currents, so
blocking one is usually tolerated; against a reduced-reserve background the *same* drug
block is far more often torsadogenic. For a borderline drug (**ranolazine**), every
LQTS background raises the susceptible fraction.

In [ ]:
healthy = assess_population(ds, "ranolazine", population="illustrative_v0",
                           n_models=60, seed=0).susceptible_fraction
print(f"healthy           susceptible: {healthy:.0%}")
for pop_id in ("lqt1", "lqt2", "lqt3"):
    a = assess_population(ds, "ranolazine", population=pop_id, n_models=60, seed=0)
    print(f"{pop_id} {str(a.conductance_scale):>16}  susceptible: {a.susceptible_fraction:.0%}")
    # reduced repolarization reserve => the same drug is more often high-risk
    assert a.susceptible_fraction > healthy
    assert a.tier == "D"
    assert "never a per-patient/genotype claim" in a.summary()

The reduced kernel reproduces the *textbook* mechanism without a tuned number: the
LQTS mean shifts (`lqt2` = `{IKr: 0.5}`, etc.) are illustrative heterozygous-scale
values, and the qNet/APD thresholds stay the **healthy** reference — so what moves the
susceptibility is the mechanism, not a fitted parameter. It remains Tier D.

## 3. Experimentally-calibrated population (Britton et al. 2013)

The v0.1/v0.3 population accepts *every* conductance draw — including the implausible
tail where an extreme combination yields a drug-free AP no real myocyte would show.
The [Britton 2013](https://doi.org/10.1073/pnas.1304382110) discipline calibrates the
population by **rejecting any candidate whose drug-free AP biomarkers fall outside
accepted ranges**. `calibrate_population` returns the accepted set and the acceptance
bookkeeping.

In [ ]:
cal = calibrate_population(ds, "calibrated_v0", n_models=80, seed=0)
print(f"accepted {len(cal.multipliers)}/{cal.n_candidates} candidates "
      f"({cal.acceptance_rate:.0%}); rejected by biomarker:")
for k, v in cal.rejection_reasons.items():
    print(f"  {k:18} {v}")

# Rejection genuinely happens (the raw prior has an abnormal tail) ...
assert cal.n_candidates > len(cal.multipliers)
assert cal.acceptance_rate < 1.0
# ... and the long/triangular-repolarization bound is the dominant filter, which is
# mechanistically right (high drug-free triangulation is itself an instability marker).
assert cal.rejection_reasons["triangulation_ms"] >= cal.rejection_reasons["vrest_mv"]

In [ ]:
# Assessing a drug against the calibrated population: still a distribution, still Tier D.
calp = assess_population(ds, "dofetilide", population="calibrated_v0", n_models=60, seed=0)
print(calp.summary())
assert calp.calibrated
assert calp.tier == "D"
assert "NOT FOR PREDICTION" in calp.summary()
# the calibration buys physiological plausibility, NOT a fit to patient data
assert "KERNEL-plausibility" in calp.summary() or "kernel-plausibility" in calp.summary().lower()

*The population layers spread a drug's risk over plausible hearts — a variability
cloud, a disease-shifted background, or a drug-free-plausibility-calibrated cohort —
and report a susceptible fraction with its acceptance bookkeeping. Like every
Harmonia output it is a distribution, and here, explicitly, **calibrated to
plausibility and never to prediction** (Tier D). See
[`docs/specs/v0.3-disease-populations.md`](../docs/specs/v0.3-disease-populations.md)
and [`docs/specs/v0.5-calibrated-populations.md`](../docs/specs/v0.5-calibrated-populations.md).*